# User History & Memory for Small Language Models
## Teaching a Small LM to "Know" Its Users

This notebook explores how to give a small language model the *appearance* of learning about a user over time, without modifying the model's weights. The trick is **memory management**: storing what the user has shared, deciding what's important, compacting it to fit in a limited context window, and injecting it as a system message.

### What this notebook covers:

1. **Key-Value (KV) Memory** — a structured dictionary of facts extracted from the conversation
2. **Vector Embedding Memory** — a similarity-search store that retrieves the most *relevant* past turns for a given query
3. **20 Realistic User Prompts** — a scripted conversation that gradually reveals who the user is
4. **Memory Compaction** — using the model itself to summarise what it knows, fitting inside the context window
5. **Personalised Responses** — comparing the model's answers with and without the memory system message

### Why does this matter for small models?

Small models (1–3B parameters) have **limited context windows** (typically 2 048–4 096 tokens). Stuffing the entire chat history into every prompt quickly becomes prohibitive. A compact, well-curated memory profile keeps the context small while preserving the information the model needs to respond helpfully.

| Memory type | Storage | Retrieval | Best for |
|-------------|---------|-----------|----------|
| KV (dict) | Key: category, Value: fact list | Direct lookup | Structured facts (name, age, job) |
| Vector store | Raw text + embedding | Cosine similarity | Relevant past turns for a query |
| Compact summary | Plain text paragraph | Included in system message | Efficient personalisation |

### Attribution

Notebook developed in the SmallLM series by Eric Van Dusen and the Data Science Modules team at UC Berkeley.

## 1. Environment Setup

We need two packages:
- **llama-cpp-python** — to load and run the local GGUF model (same as in `LlamaCpp_SmallLM_Demo.ipynb`)
- **numpy** — for the vector similarity calculations in our embedding memory

_No external vector-database library is required — we implement a lightweight cosine-similarity store from scratch using only NumPy._

In [ ]:
# Install / import dependencies
try:
    from llama_cpp import Llama
    import numpy as np
except ImportError:
    %pip install llama-cpp-python numpy
    from llama_cpp import Llama
    import numpy as np

import re, json
print("✓ Dependencies ready")

### 1.1 Locate your model files

Choose the approach that matches your computing environment. This is the same setup as in the other SmallLM notebooks.

### Approach 1 — Shared Hub

In [ ]:
# On Cal-ICOR workshop hub (JupyterCon Nov 2025)
!ls /home/jovyan/shared/

### Approach 2 — Local machine

In [ ]:
#This is my local path to a directory called shared-rw
!ls shared-rw

In [ ]:
# or the full path (example: laptop)
!ls /Users/ericvandusen/Documents/GitHub/SmallLM-SP25/shared-rw

### 1.2 Set the model directory

In [ ]:
# Hub path (Cal-ICOR)
model_directory = "/home/jovyan/shared/"

In [ ]:
# Local path — uncomment and edit as needed
#model_directory = "/Users/ericvandusen/Documents/GitHub/SmallLM-SP25/shared-rw"

### 1.3 Load the small model

We use **Qwen2-1.5B-Instruct** (quantised to 4-bit GGUF) — the same model as the other SmallLM notebooks.  
We set a larger context window (`n_ctx=4096`) so the model can handle slightly longer memory-enriched prompts.

> **Note**: Loading may take a few seconds. Verbose output from llama.cpp is expected.

In [ ]:
import os

model_name = "qwen2-1_5b-instruct-q4_0.gguf"
model_path = os.path.join(model_directory, model_name)

model = Llama(
    model_path=model_path,
    n_ctx=4096,          # larger context to fit memory summaries
    n_threads=None,      # auto-detect
    verbose=True,
    chat_format="chatml" # Qwen uses ChatML format
)

print(f"\n✓ Model loaded: {model_name}")

---
## 2. Memory Architecture

Before writing any code it helps to understand the two complementary memory systems we'll build.

### 2a. Key-Value (KV) Memory

KV memory is like a personal fact-sheet. Every time the user reveals something about themselves, we extract the key fact and store it under a category label:

```
{
  "NAME":     ["Maya Patel"],
  "JOB":      ["machine learning engineer at a biotech company"],
  "HOBBY":    ["rock climbing", "cooking elaborate meals"],
  "LOCATION": ["Boston"],
  ...
}
```

**Advantages**: Very compact, easy to read, direct lookup.  
**Disadvantage**: Requires a fact-extraction step; nuance can be lost.

### 2b. Vector Embedding Memory

Vector memory stores each conversation turn as a text snippet *and* a numerical vector that captures the semantic content. When a new question arrives we compute its vector and find the stored turns with the highest cosine similarity — returning only the most *relevant* memories for the current context.

```
Stored turn:  "I love rock climbing - I go twice a week"
Query:        "What sports do you enjoy?"
→ High similarity → retrieved into context
```

We implement this with a simple **bag-of-words TF vector** and NumPy cosine similarity — no external vector DB needed.

**Advantages**: Retrieves relevant context dynamically; scales well.  
**Disadvantage**: Bag-of-words misses synonyms; a real system would use dense embeddings.

### 2c. The memory pipeline

```
User turn
   │
   ├─► KVMemory.add_turn()       → extract & store structured facts
   └─► VectorMemory.add()        → store turn as searchable vector
                                         │
                              At query time:
                                         │
                              compact_summary = KV facts (all)
                              relevant_turns  = VectorMemory.search(query)
                                         │
                              Inject both into system message
                                         │
                              model.create_chat_completion()
```

---
## 3. Key-Value Memory

The `KVMemory` class is a thin wrapper around a Python dictionary.  
It stores facts by category and can produce a compact bullet-list summary.

In [ ]:
class KVMemory:
    """A simple key-value fact store for user profile information."""

    def __init__(self):
        self.facts = {}          # category (str) → list[str]
        self.turn_history = []   # raw (user, assistant) turn log

    # ── fact management ─────────────────────────────────────────────

    def add_fact(self, category: str, fact: str):
        """Add a single fact under a category (duplicate-safe)."""
        category = category.upper().strip()
        fact = fact.strip()
        if not fact:
            return
        if category not in self.facts:
            self.facts[category] = []
        if fact not in self.facts[category]:   # avoid exact duplicates
            self.facts[category].append(fact)

    def add_facts_dict(self, facts_dict: dict):
        """Merge a {category: [fact, ...]} dict into the store."""
        for category, fact_list in facts_dict.items():
            for fact in (fact_list if isinstance(fact_list, list) else [fact_list]):
                self.add_fact(category, fact)

    def add_turn(self, user_message: str, assistant_response: str = ""):
        """Log a raw conversation turn."""
        self.turn_history.append({"user": user_message, "assistant": assistant_response})

    # ── retrieval ────────────────────────────────────────────────────

    def get_all_facts(self) -> list:
        """Return all stored facts as a flat list of 'CATEGORY: fact' strings."""
        result = []
        for category, facts in sorted(self.facts.items()):
            for fact in facts:
                result.append(f"{category}: {fact}")
        return result

    def get_summary_text(self) -> str:
        """Return all facts as a newline-separated string."""
        lines = self.get_all_facts()
        return "\n".join(lines) if lines else "(no facts stored yet)"

    def get_facts_for_category(self, category: str) -> list:
        return self.facts.get(category.upper(), [])

    # ── display ──────────────────────────────────────────────────────

    def __repr__(self):
        n_facts = sum(len(v) for v in self.facts.values())
        return f"KVMemory({len(self.facts)} categories, {n_facts} facts, {len(self.turn_history)} turns)"

    def display(self):
        print(self)
        for cat, facts in sorted(self.facts.items()):
            print(f"  [{cat}]")
            for f in facts:
                print(f"    • {f}")

### 3a. Demo — manually adding facts

Let's see how the store works with a few hand-crafted facts.

In [ ]:
kv_mem = KVMemory()

# Add some facts manually
kv_mem.add_fact("NAME", "Maya Patel")
kv_mem.add_fact("AGE", "31")
kv_mem.add_fact("JOB", "machine learning engineer")
kv_mem.add_fact("HOBBY", "rock climbing")
kv_mem.add_fact("HOBBY", "cooking")
kv_mem.add_fact("LOCATION", "Boston")

# Try adding a duplicate — it should be ignored
kv_mem.add_fact("HOBBY", "rock climbing")

kv_mem.display()

print("\n--- Summary text ---")
print(kv_mem.get_summary_text())

---
## 4. Vector Embedding Memory

The `VectorMemory` class stores text snippets together with their **bag-of-words TF vectors**.  
Retrieval uses **cosine similarity** — a query is vectorised the same way and we return the top-*k* most similar stored snippets.

> **Why bag-of-words instead of a neural embedding?**  
> It requires only NumPy (already installed), has no API costs, and is transparent enough to understand in a teaching notebook. In production you would use a sentence-transformer model or an embedding API to get much better semantic coverage.

Cosine similarity between vectors **u** and **v**:

$$\text{sim}(u, v) = \frac{u \cdot v}{\|u\| \|v\|}$$

In [ ]:
class VectorMemory:
    """
    A minimal vector store backed by bag-of-words TF vectors and NumPy cosine similarity.
    Suitable for demonstration; replace the _vectorize() method with a sentence-transformer
    for production-quality semantic search.
    """

    def __init__(self):
        self._texts   = []   # list[str]       — raw stored snippets
        self._vectors = []   # list[np.ndarray] — corresponding TF vectors
        self._vocab   = {}   # word → index in vector

    # ── internals ────────────────────────────────────────────────────

    @staticmethod
    def _tokenize(text: str) -> list:
        return re.findall(r'\b[a-z]+\b', text.lower())

    def _rebuild_vocab(self):
        """Rebuild the shared vocabulary from all stored texts."""
        all_words = set()
        for text in self._texts:
            all_words.update(self._tokenize(text))
        self._vocab = {word: idx for idx, word in enumerate(sorted(all_words))}

    def _vectorize(self, text: str) -> np.ndarray:
        """Term-frequency vector over the shared vocabulary."""
        tokens = self._tokenize(text)
        vec = np.zeros(len(self._vocab), dtype=float)
        for token in tokens:
            if token in self._vocab:
                vec[self._vocab[token]] += 1.0
        norm = np.linalg.norm(vec)
        return vec / norm if norm > 0 else vec

    # ── public API ───────────────────────────────────────────────────

    def add(self, text: str):
        """Add a new snippet and rebuild all vectors to use the updated vocabulary."""
        self._texts.append(text)
        self._rebuild_vocab()
        self._vectors = [self._vectorize(t) for t in self._texts]

    def search(self, query: str, top_k: int = 3, min_score: float = 0.05) -> list:
        """
        Return the top-k stored snippets most similar to `query`.
        Snippets with cosine similarity below `min_score` are excluded.
        """
        if not self._texts:
            return []
        q_vec = self._vectorize(query)
        if np.linalg.norm(q_vec) == 0:
            return []
        scores = [
            (float(np.dot(q_vec, v)), t)
            for v, t in zip(self._vectors, self._texts)
        ]
        scores.sort(reverse=True)
        return [t for s, t in scores[:top_k] if s >= min_score]

    def __repr__(self):
        return f"VectorMemory({len(self._texts)} snippets, vocab size {len(self._vocab)})"

### 4a. Demo — adding turns and searching

Let's add several sentences about a hypothetical user and then query for relevant ones.

In [ ]:
vec_mem = VectorMemory()

sample_turns = [
    "My name is Maya and I'm 31 years old.",
    "I work as a machine learning engineer at a biotech company in Boston.",
    "I love rock climbing and go to the gym twice a week.",
    "I've been learning Portuguese for about a year.",
    "My favourite food is Ethiopian — especially injera with doro wat.",
    "I got my PhD in Computer Science from MIT.",
    "I have two rescue cats named Pixel and Byte.",
]

for turn in sample_turns:
    vec_mem.add(turn)

print(vec_mem)
print()

queries = [
    "What sports or exercise does the user enjoy?",
    "Where did the user go to school?",
    "Does the user have any pets?",
    "What does the user eat?",
]

for q in queries:
    results = vec_mem.search(q, top_k=2)
    print(f"Query: {q}")
    for r in results:
        print(f"  → {r}")
    print()

---
## 5. Extracting Facts with the Small Model

So far we have been inserting facts **manually**. In a real system the model itself should read each user turn and extract the key facts automatically.

The function below crafts a short extraction prompt, calls the model, and parses the structured response into a `{category: [fact]}` dictionary that can be fed directly into `KVMemory.add_facts_dict()`.

**Supported categories:**

| Category | Examples |
|----------|----------|
| NAME | "Maya Patel" |
| AGE | "31" |
| LOCATION | "Boston" |
| JOB | "ML engineer at biotech" |
| HOBBY | "rock climbing" |
| FAMILY | "two rescue cats" |
| FOOD | "Ethiopian food" |
| TRAVEL | "visited Japan" |
| EDUCATION | "PhD from MIT" |
| GOAL | "run a half-marathon" |
| CHALLENGE | "managing a team for the first time" |
| LANGUAGE | "learning Portuguese" |
| OTHER | anything else worth noting |

In [ ]:
VALID_CATEGORIES = {
    "NAME", "AGE", "LOCATION", "JOB", "HOBBY",
    "FAMILY", "FOOD", "TRAVEL", "EDUCATION", "GOAL",
    "CHALLENGE", "LANGUAGE", "OTHER"
}


def extract_facts(model, user_message: str, max_tokens: int = 120) -> dict:
    """
    Ask the model to extract structured facts from a single user turn.
    Returns a dict {CATEGORY: [fact_string, ...]}.
    """
    extraction_prompt = (
        "Extract personal facts about the user from the message below. "
        "Output ONLY lines in the format  CATEGORY: fact  (one per line). "
        "Use these categories: NAME, AGE, LOCATION, JOB, HOBBY, FAMILY, FOOD, "
        "TRAVEL, EDUCATION, GOAL, CHALLENGE, LANGUAGE, OTHER. "
        "If there are no facts, output NONE.\n\n"
        f'User message: "{user_message}"\n\nFacts:'
    )

    response = model.create_chat_completion(
        messages=[{"role": "user", "content": extraction_prompt}],
        max_tokens=max_tokens,
        temperature=0.1,   # low temperature → more consistent structured output
    )
    raw = response["choices"][0]["message"]["content"]
    return _parse_facts(raw)


def _parse_facts(raw_text: str) -> dict:
    """Parse 'CATEGORY: fact' lines into a dict."""
    result = {}
    for line in raw_text.strip().splitlines():
        if ":" not in line:
            continue
        cat, _, fact = line.partition(":")
        cat  = cat.strip().upper()
        fact = fact.strip()
        if cat in VALID_CATEGORIES and fact and fact.upper() != "NONE":
            result.setdefault(cat, []).append(fact)
    return result

### 5a. Test fact extraction on a single message

We run a single turn through the extraction pipeline to verify it works before processing all 20 prompts.

In [ ]:
test_message = (
    "Hi! I'm Maya Patel, I'm 31 and I work as a machine learning engineer "
    "at a biotech startup in Boston. In my spare time I love rock climbing."
)

extracted = extract_facts(model, test_message)

print("Extracted facts:")
for cat, facts in extracted.items():
    for fact in facts:
        print(f"  {cat}: {fact}")

---
## 6. Building a User Profile from 20 Prompts

Below are 20 messages from a single user ("Maya Patel") that span several topics.  
Each message reveals one or two new facts.  
We process every message through **both** memory stores:

1. **KVMemory** — via the model's fact-extraction call
2. **VectorMemory** — the raw message text is stored as-is

After all 20 messages we will have a rich profile ready for compaction.

In [ ]:
user_prompts = [
    # Identity
    "Hi there! My name is Maya Patel and I'm 31 years old.",
    "I work as a machine learning engineer at a biotech company in Boston.",
    "I grew up in Chicago but moved to Boston for grad school and stayed.",
    # Interests
    "My biggest hobby is rock climbing — I go to the gym twice a week and do outdoor climbs on weekends.",
    "I love cooking elaborate weekend meals. Last Saturday I spent all day making a Moroccan tagine.",
    "I'm a huge sci-fi reader — Ted Chiang and Liu Cixin are my favourite authors.",
    "My morning routine includes 30 minutes of yoga and journaling.",
    # Family / relationships
    "I have two rescue cats named Pixel and Byte.",
    "My partner and I just adopted a puppy named Waffles last month — it's chaotic but wonderful.",
    # Food / lifestyle
    "I'm obsessed with Ethiopian food, especially injera with doro wat.",
    "I've been doing intermittent fasting for two years and it's really changed how I feel.",
    # Education
    "I got my PhD in Computer Science from MIT, focusing on neural architecture search.",
    "I'm working on publishing my first research paper on efficient model fine-tuning.",
    # Travel / languages
    "I visited Japan twice and lived in Berlin for a summer during college.",
    "I've been learning Portuguese for about a year because I want to spend a month in Brazil.",
    # Goals
    "I'm hoping to run a half-marathon before I turn 35.",
    "My goal this year is to meditate every day and read at least 20 books.",
    # Challenges
    "My biggest professional challenge right now is managing a team for the first time — it's harder than I expected.",
    "I sometimes feel overwhelmed by the pace of AI development — it's exciting but a bit scary.",
    # Future plans
    "I'm seriously thinking about leaving my company to start my own AI startup focused on healthcare.",
]

print(f"Total prompts: {len(user_prompts)}")

### 6a. Process all 20 prompts through both memory stores

For each message we:
1. Add the raw text to `VectorMemory`
2. Call the model to extract structured facts → add to `KVMemory`

> This may take a couple of minutes because the model runs a short extraction call for each of the 20 prompts.

In [ ]:
# Initialise both memory stores
kv_mem  = KVMemory()
vec_mem = VectorMemory()

print("Processing prompts...")
print("-" * 60)

for i, prompt in enumerate(user_prompts, start=1):
    # 1) Store raw text in vector memory
    vec_mem.add(prompt)

    # 2) Extract structured facts with the model
    facts = extract_facts(model, prompt)

    # 3) Merge extracted facts into KV store
    kv_mem.add_facts_dict(facts)

    # 4) Log the raw turn
    kv_mem.add_turn(prompt)

    # Progress indicator
    fact_str = ", ".join(f"{k}: {v[0]}" for k, v in facts.items()) if facts else "(none extracted)"
    print(f"[{i:02d}] {fact_str}")

print("-" * 60)
print(f"\nDone! {kv_mem}")
print(vec_mem)

### 6b. Inspect the accumulated memory

Let's look at the full KV fact store and try a few vector searches.

In [ ]:
print("=" * 60)
print("KV Memory — full fact store")
print("=" * 60)
kv_mem.display()

print()
print("=" * 60)
print("Vector Memory — similarity search demo")
print("=" * 60)

demo_queries = [
    "What kind of work does the user do?",
    "What are the user's fitness and health habits?",
    "What are the user's future plans?",
]

for q in demo_queries:
    hits = vec_mem.search(q, top_k=2)
    print(f"\nQuery: {q}")
    for h in hits:
        print(f"  → {h}")

---
## 7. Memory Compaction

### Why compaction?

Our small model has a context window of 4 096 tokens. A rough rule of thumb is **~0.75 tokens per word** in English.  
The 20 raw prompts above total around **500 words ≈ 375 tokens** — manageable now, but in a real application with hundreds of turns, raw history would quickly overflow.

**Compaction** uses the model itself to distil the collected facts into a terse, coherent paragraph — the *user profile* — that we can safely include in every system message.

| Form | Approx. tokens |
|------|---------------|
| All 20 raw turns | ~375 |
| KV flat list | ~120 |
| Compact summary | ~80 |

The compact summary is what we actually inject into the system message.

In [ ]:
def compact_memory(model, kv_memory: KVMemory, max_tokens: int = 200) -> str:
    """
    Ask the model to write a concise 3–5 sentence user profile
    from the accumulated KV facts.
    Returns the compact profile as a plain-text string.
    """
    facts_text = kv_memory.get_summary_text()

    if facts_text == "(no facts stored yet)":
        return "No user information available yet."

    prompt = (
        "You are a helpful assistant building a personalised user profile. "
        "Given the following raw facts about a user, write a concise 3–5 sentence "
        "summary that captures the most important and distinctive information. "
        "Be specific, factual, and natural — write in third person.\n\n"
        f"Raw facts:\n{facts_text}\n\nCompact profile (3–5 sentences):"
    )

    response = model.create_chat_completion(
        messages=[{"role": "user", "content": prompt}],
        max_tokens=max_tokens,
        temperature=0.3,
    )
    return response["choices"][0]["message"]["content"].strip()

In [ ]:
# Generate the compact user profile
compact_profile = compact_memory(model, kv_mem)

print("Compact user profile:")
print("-" * 60)
print(compact_profile)
print("-" * 60)

# Rough token estimate: 0.75 tokens per word
raw_words     = sum(len(t.split()) for t in user_prompts)
profile_words = len(compact_profile.split())

print(f"\nRaw turns:          ~{int(raw_words * 0.75):>4} tokens ({raw_words} words)")
print(f"KV flat list:       ~{int(len(kv_mem.get_all_facts()) * 8):>4} tokens (estimated)")
print(f"Compact profile:    ~{int(profile_words * 0.75):>4} tokens ({profile_words} words)")

---
## 8. Personalised Chat — With and Without Memory

Now comes the payoff. We compare two ways of answering the same question:

| Mode | System message |
|------|---------------|
| **Without memory** | Generic "you are a helpful assistant" |
| **With KV memory** | Compact user profile injected at the top |
| **With vector + KV** | Compact profile **plus** the most relevant past turns for the query |

The model's weights do not change — only the context does. Yet the responses appear strikingly more personal.

In [ ]:
def chat_without_memory(model, user_message: str, max_tokens: int = 200) -> str:
    """Answer a question with no user context — the baseline."""
    messages = [
        {"role": "system",  "content": "You are a helpful AI assistant."},
        {"role": "user",    "content": user_message},
    ]
    response = model.create_chat_completion(
        messages=messages, max_tokens=max_tokens, temperature=0.7
    )
    return response["choices"][0]["message"]["content"]


def chat_with_kv_memory(model, user_message: str,
                        compact_profile: str,
                        max_tokens: int = 200) -> str:
    """
    Answer a question with the compact user profile in the system message.
    This is the lightweight personalisation approach.
    """
    system_message = (
        "You are a helpful AI assistant. "
        "You know the following about the user:\n\n"
        f"{compact_profile}\n\n"
        "Use this context to personalise your response where relevant. "
        "Be natural — do not explicitly mention that you looked up stored facts."
    )
    messages = [
        {"role": "system", "content": system_message},
        {"role": "user",   "content": user_message},
    ]
    response = model.create_chat_completion(
        messages=messages, max_tokens=max_tokens, temperature=0.7
    )
    return response["choices"][0]["message"]["content"]


def chat_with_full_memory(model, user_message: str,
                          compact_profile: str,
                          vec_memory: VectorMemory,
                          max_tokens: int = 250) -> str:
    """
    Answer a question using both:
    - The compact KV profile (always included)
    - The top-3 most relevant raw turns retrieved by vector similarity
    """
    relevant = vec_memory.search(user_message, top_k=3)
    relevant_text = "\n".join(f"- {r}" for r in relevant) if relevant else "(none)"

    system_message = (
        "You are a helpful AI assistant. "
        "You know the following about the user:\n\n"
        f"{compact_profile}\n\n"
        "Relevant excerpts from past conversation:\n"
        f"{relevant_text}\n\n"
        "Use this context to personalise your response where relevant. "
        "Be natural — do not mention that you are using stored information."
    )
    messages = [
        {"role": "system", "content": system_message},
        {"role": "user",   "content": user_message},
    ]
    response = model.create_chat_completion(
        messages=messages, max_tokens=max_tokens, temperature=0.7
    )
    return response["choices"][0]["message"]["content"]

### 8a. Side-by-side comparison

Run each of the three functions on the same question and compare the outputs.  
Notice how the memory-aware responses refer to specific details about Maya.

In [ ]:
comparison_questions = [
    "Can you recommend a book I might enjoy?",
    "I need some motivation — what should I focus on this week?",
    "What career advice do you have for me?",
]

for question in comparison_questions:
    print("\n" + "=" * 70)
    print(f"Question: {question}")
    print("=" * 70)

    ans_baseline = chat_without_memory(model, question)
    ans_kv       = chat_with_kv_memory(model, question, compact_profile)
    ans_full     = chat_with_full_memory(model, question, compact_profile, vec_mem)

    print("\n[No memory]")
    print(ans_baseline)

    print("\n[KV compact profile]")
    print(ans_kv)

    print("\n[KV profile + vector retrieval]")
    print(ans_full)

### 8b. Simulating a longer conversation with growing context

Here we re-generate the compact profile after each turn, simulating an ongoing conversation where the model's knowledge of the user keeps improving.

In [ ]:
# Re-initialise fresh memory stores for a clean demo
conv_kv  = KVMemory()
conv_vec = VectorMemory()

# Conversation: 5 turns that reveal info, then a personalised question
conversation_turns = [
    "Hi, I'm Alex — a 25-year-old software engineer living in Seattle.",
    "I'm really into coffee — I just got into pour-over brewing.",
    "I've been learning Japanese for 6 months because I want to visit Kyoto.",
    "My goal is to run a 5K by the end of the year.",
    "I mostly eat plant-based food but I'm not super strict about it.",
]

final_question = "Any suggestions for how I can make the most of a trip to Japan?"

print("Building memory across conversation turns...\n")

for i, turn in enumerate(conversation_turns, start=1):
    conv_vec.add(turn)
    facts = extract_facts(model, turn)
    conv_kv.add_facts_dict(facts)
    conv_kv.add_turn(turn)
    print(f"Turn {i}: {turn}")
    if facts:
        print(f"  Extracted: {facts}")

# Compact the accumulated memory
conv_profile = compact_memory(model, conv_kv)

print("\n" + "-" * 60)
print("Compact profile after 5 turns:")
print(conv_profile)
print("-" * 60)

print(f"\nQuestion: {final_question}")
print("\n[No memory]")
print(chat_without_memory(model, final_question, max_tokens=150))
print("\n[Full memory]")
print(chat_with_full_memory(model, final_question, conv_profile, conv_vec, max_tokens=150))

---
## 9. Persisting Memory to Disk

A real application needs memory that survives between sessions. We can serialise the KV store to a JSON file and reload it at startup.

> **Note**: `VectorMemory` stores vectors that depend on a shared vocabulary built from all seen texts. The simplest persistence strategy is to re-add all stored texts from a saved list on reload, which rebuilds the vocabulary and vectors automatically.

In [ ]:
import json, pathlib


def save_memory(kv_memory: KVMemory, filepath: str):
    """Persist KV facts and raw turn history to a JSON file."""
    data = {
        "facts": kv_memory.facts,
        "turn_history": kv_memory.turn_history,
    }
    with open(filepath, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)
    print(f"✓ Memory saved to {filepath}")


def load_memory(filepath: str) -> tuple:
    """
    Load a previously saved memory file.
    Returns (kv_memory, vec_memory) ready for use.
    """
    with open(filepath, encoding="utf-8") as f:
        data = json.load(f)

    kv = KVMemory()
    kv.facts        = data.get("facts", {})
    kv.turn_history = data.get("turn_history", [])

    vec = VectorMemory()
    for turn in kv.turn_history:
        vec.add(turn["user"])   # rebuild vectors from saved texts

    print(f"✓ Memory loaded from {filepath}: {kv}")
    return kv, vec


# Demo: save and reload the memory we built in Section 6
memory_file = "user_memory_maya.json"

save_memory(kv_mem, memory_file)

kv_reloaded, vec_reloaded = load_memory(memory_file)
kv_reloaded.display()

---
## Summary

In this notebook you learned how to give a small local language model the *appearance* of remembering and learning about a user — without changing the model weights at all.

### What we built

| Component | Class / Function | Purpose |
|-----------|-----------------|--------|
| KV Memory | `KVMemory` | Stores structured facts by category |
| Vector Memory | `VectorMemory` | Retrieves relevant past turns by cosine similarity |
| Fact extraction | `extract_facts()` | Uses the model to parse facts from each user turn |
| Compaction | `compact_memory()` | Distils all facts into a short profile paragraph |
| Personalised chat | `chat_with_kv_memory()` / `chat_with_full_memory()` | Injects the profile into the system message |
| Persistence | `save_memory()` / `load_memory()` | JSON serialisation of the memory store |

### Key ideas

1. **LLMs have no intrinsic memory** — memory is always managed by the application layer, not the model.
2. **KV memory** is fast and readable; **vector memory** is flexible and query-driven. Both are complementary.
3. **Context-window budget matters** — a compact profile (~80 tokens) is far more practical than raw history (~375 tokens) once conversations grow long.
4. **The model both writes and reads memory** — it extracts facts *and* later uses the compact summary to personalise responses. The same small model does both jobs.

### Extensions to explore

- Replace the bag-of-words vectoriser with a **sentence-transformer** model (e.g., `all-MiniLM-L6-v2` via `sentence-transformers`) for better semantic retrieval.
- Swap the flat JSON store for a proper **vector database** like [ChromaDB](https://www.trychroma.com/) or [FAISS](https://github.com/facebookresearch/faiss).
- Add a **forgetting mechanism** — age out facts that are rarely retrieved.
- Combine this notebook with the **Gradio** chatbot notebooks to build a full interactive demo with persistent memory.
- Explore **LangChain's `ConversationSummaryMemory`** as a higher-level alternative to the hand-rolled compaction function.